In [4]:
#Task 3.1 

import sqlite3 

connection = sqlite3.connect("minesweeper.db") 

connection.execute('''CREATE TABLE IF NOT EXISTS Board(
                        xCoordinate INTEGER,
                        yCoordinate INTEGER,
                        hasMine INTEGER,
                        checked INTEGER DEFAULT 0,
                        PRIMARY KEY (xCoordinate, yCoordinate))''') 

connection.commit() 

connection.close() 

In [ ]:
#Task 3.2 

import socket, sqlite3, random 

print("-------------------------")
print("MINESWEEPER GAME [SERVER]")
print("-------------------------")
print()

my_socket = socket.socket()
my_socket.bind(('127.0.0.1', 6789))

my_socket.listen(5) 

#Establish connection to minesweeper.db 
connection = sqlite3.connect("minesweeper.db") 

#Delete all records in Board Table 
connection.execute('''DELETE FROM Board''') 

#Insert 16 records 
has_mine = [] 

board_info = [] 

#Randomly select 10 coor to have a mine 
while len(has_mine) < 10: 
    valid = False 
    while valid == False: 
        x_coor = random.randint(1,4) 
        y_coor = random.randint(1,4) 
        
        if (x_coor,y_coor) not in has_mine: 
            has_mine.append((x_coor,y_coor)) 
            valid = True 

#Finding the information for each coordinate 
for i in range(1,5): 
    for j in range(1,5): 
        if (i,j) not in has_mine: 
            board_info.append([i,j,0,0]) 
        else: 
            board_info.append([i,j,1,0]) 

#Insert the 16 records into the database 
connection.executemany('''INSERT INTO Board(xCoordinate,yCoordinate,hasMine,checked) VALUES (?,?,?,?)''',board_info) 

connection.commit() 

#Wait for connection from client program
print("Waiting for player to join") 
print("--------------------------")

c,addr = my_socket.accept() 

checked_count = 0 

while True: 
    
    #Collect user input for x and y coordinate 
    coordinate = c.recv(1024).decode() 
    x_coor,y_coor = int(coordinate.split(",")[0]),int(coordinate.split(",")[1])  
    print(f"User x coordinate: {x_coor}, User y coordinate: {y_coor}")

    #Update the checked value of the selected coordinate 
    connection.execute('''UPDATE Board SET checked = 1 WHERE xCoordinate=? AND yCoordinate = ?''',(x_coor,y_coor))
    connection.commit() 

    checked_count += 1 
    
    #Obtain the hasmine value for the selected coordinate 
    cur = connection.cursor() 

    cur.execute('''SELECT hasMine FROM Board WHERE xCoordinate=? AND yCoordinate = ?''',(x_coor,y_coor))

    hasMine_result = cur.fetchone()[0] 

    #Situation when user hits a mine 
    if hasMine_result == 1: 
        print("Hit MINE") 
        c.sendall("MINE".encode()) 
        break 

    else: 


        #Find for the number of surrounding mines
        possible_surrounding = [
            (x_coor+1,y_coor),
            (x_coor-1,y_coor),
            (x_coor,y_coor+1),
            (x_coor,y_coor-1),
            (x_coor+1,y_coor+1),
            (x_coor-1,y_coor+1),
            (x_coor+1,y_coor-1),
            (x_coor-1,y_coor-1)
        ] 

        moves = [
            (1,0),
            (-1,0),
            (0,1),
            (0,-1),
            (1,1),
            (-1,1),
            (1,-1),
            (-1,-1)
        ]

        possible_surrounding = [
            (x_coor+i,y_coor+j) for (i,j) in moves
        ]

        valid_surrounding = [] 
        
        for coor in possible_surrounding: 
            if 1 <= coor[0] <= 4 and 1<= coor[1] <= 4: 
                valid_surrounding.append(coor) 

        no_of_surrounding_mine = 0 

        for coor in valid_surrounding: 
            cur.execute('''SELECT hasMine FROM Board WHERE xCoordinate = ? AND yCoordinate = ?''',(coor[0],coor[1]))

            hasMine_result = cur.fetchone()[0] 

            if hasMine_result == 1: 
                no_of_surrounding_mine += 1 


        print(f"no_of_surrounding_mines: {no_of_surrounding_mine}") 
        print(f"no_of_coordinates_checked: {checked_count}") 
        print("\n") 

        #Send response to client 
        response = str(no_of_surrounding_mine) + "," + str(checked_count) 

        c.sendall(response.encode()) 

        #Check for the condition where the game ends 
        if checked_count == 6: 
            break 

connection.close() 
c.close() 
my_socket.close() 
 
print("-------------------")
print("GAME ENDED")
print("-------------------")    



-------------------------
MINESWEEPER GAME [SERVER]
-------------------------

Waiting for player to join
--------------------------
User x coordinate: 1, User y coordinate: 1
no_of_surrounding_mines: 1
no_of_coordinates_checked: 1


User x coordinate: 2, User y coordinate: 1
no_of_surrounding_mines: 3
no_of_coordinates_checked: 2


User x coordinate: 2, User y coordinate: 2
no_of_surrounding_mines: 5
no_of_coordinates_checked: 3


User x coordinate: 3, User y coordinate: 3
no_of_surrounding_mines: 6
no_of_coordinates_checked: 4


User x coordinate: 4, User y coordinate: 1
no_of_surrounding_mines: 3
no_of_coordinates_checked: 5


User x coordinate: 4, User y coordinate: 3
no_of_surrounding_mines: 4
no_of_coordinates_checked: 6


-------------------
GAME ENDED
-------------------


In [1]:
%run TASK4_TanYuYing_3014_0155

 * Serving Flask app 'TASK4_TanYuYing_3014_0155'
 * Debug mode: off


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
